In [5]:
import pandas as pd
import numpy as np

# Load Austria data with growth variables
df = pd.read_pickle('/Users/aryan.bery/Desktop/WHU/Data Driven Entrepreneurship/austria_dashboard_assignment/data/processed_copy/austria_data_with_growth.pkl')
print(f"Loaded data: {df.shape}")
print(f"Columns: {len(df.columns)}")
print("Available growth columns:", [col for col in df.columns if 'growth' in col or 'aagr' in col])

FileNotFoundError: [Errno 2] No such file or directory: '/Users/aryan.bery/Desktop/WHU/Data Driven Entrepreneurship/austria_dashboard_assignment/data/processed_copy/austria_data_with_growth.pkl'

## Consistent High-Growth Firm Classification (2024)

Following the Belgium notebook logic exactly:

**Variable**: `ConsistentHighGrowthFirm_2024`

**Classification Rules**:
1. If any required growth variables are unavailable → "n.a."
2. If base-year employees (2021) < 10 → "n.a."
3. Otherwise:
   - Check if yearly growth > 20% in 2022, 2023, 2024
   - Require >20% growth in at least 2 of the 3 years
   - Require AAGR_2024 > 20%
   - If both conditions met → classify as 1 (High-Growth)
   - Otherwise → classify as 0 (Not High-Growth)

**Required Variables**:
- `growth_2022`, `growth_2023`, `growth_2024`
- `aagr_2024`
- `emp_2021_num` (for size threshold)

In [ ]:
# Create Consistent High-Growth Firm classification
# Initialize with "n.a."
df['ConsistentHighGrowthFirm_2024'] = 'n.a.'

# Check for missing required variables
required_growth_vars = ['growth_2022', 'growth_2023', 'growth_2024', 'aagr_2024']
missing_mask = (
    (df['growth_2022'] == 'n.a.') |
    (df['growth_2023'] == 'n.a.') |
    (df['growth_2024'] == 'n.a.') |
    (df['aagr_2024'] == 'n.a.')
)

# Check for size threshold (emp_2021 < 10)
size_mask = df['emp_2021_num'] < 10

# Combined exclusion mask
exclusion_mask = missing_mask | size_mask

print(f"Firms excluded due to missing data: {missing_mask.sum()}")
print(f"Firms excluded due to size threshold (<10 employees in 2021): {size_mask.sum()}")
print(f"Total firms excluded: {exclusion_mask.sum()}")
print(f"Firms available for classification: {(~exclusion_mask).sum()}")

# For firms that pass the exclusion criteria, apply classification logic
valid_firms = ~exclusion_mask

if valid_firms.sum() > 0:
    # Convert growth variables to numeric for comparison
    growth_2022_num = pd.to_numeric(df.loc[valid_firms, 'growth_2022'], errors='coerce')
    growth_2023_num = pd.to_numeric(df.loc[valid_firms, 'growth_2023'], errors='coerce')
    growth_2024_num = pd.to_numeric(df.loc[valid_firms, 'growth_2024'], errors='coerce')
    aagr_2024_num = pd.to_numeric(df.loc[valid_firms, 'aagr_2024'], errors='coerce')

    # Check growth > 20% in each year
    growth_2022_high = growth_2022_num > 0.20
    growth_2023_high = growth_2023_num > 0.20
    growth_2024_high = growth_2024_num > 0.20

    # Count years with high growth
    high_growth_years = growth_2022_high.astype(int) + growth_2023_high.astype(int) + growth_2024_high.astype(int)

    # Check AAGR > 20%
    aagr_high = aagr_2024_num > 20

    # Classification: at least 2 years of high growth AND AAGR > 20%
    is_high_growth = (high_growth_years >= 2) & aagr_high

    # Apply classification
    df.loc[valid_firms, 'ConsistentHighGrowthFirm_2024'] = is_high_growth.astype(int)

print("\nClassification Results:")
print(f"High-Growth Firms (1): {(df['ConsistentHighGrowthFirm_2024'] == 1).sum()}")
print(f"Not High-Growth Firms (0): {(df['ConsistentHighGrowthFirm_2024'] == 0).sum()}")
print(f"Unclassified (n.a.): {(df['ConsistentHighGrowthFirm_2024'] == 'n.a.').sum()}")

# Show some examples
print("\nSample classifications:")
sample_df = df[['company_name', 'emp_2021_num', 'growth_2022', 'growth_2023', 'growth_2024', 'aagr_2024', 'ConsistentHighGrowthFirm_2024']].head(10)
print(sample_df.to_string())

NameError: name 'df' is not defined